# Reproducing Zhuang et al. (2025) from dandiset 001177

Paper: Zhuang X, Lemak J, Sridhar S, Nelson AB. Aberrant striatal firing mediates impulsive
decision-making in a mouse model of Parkinson's disease. *Brain* 149:238 (2025),
[doi:10.1093/brain/awaf312](https://doi.org/10.1093/brain/awaf312). The full text (open access,
CC BY 4.0, from Europe PMC, PMC12597312) and its Supplementary material were used.

Data: [DANDI:001177 version 0.250902.2127](https://dandiarchive.org/dandiset/001177/0.250902.2127),
streamed from the archive. This is the only published version. The paper cites the dandiset
without a version, so the most recent published version is pinned; the data check below
compares its file list with the current draft. The behavioural data, rotarod scores and
histology the paper reports are not in the dandiset; they were deposited on Zenodo
([doi:10.5281/zenodo.15258978](https://doi.org/10.5281/zenodo.15258978)) and the claims that use
them are counted apart (`data: elsewhere`).

The claims and their criteria are in `claims/001177/10.1093-brain-awaf312.yaml`.

| Claim | Data | Tested here |
|---|---|---|
| fig4bc-baseline-rates-similar | dandiset | blocked: no sorted units or optogenetic labels |
| fig4dg-ppx-bidirectional-identified | dandiset | blocked: as above |
| fig4hk-modulation-stronger-park | dandiset | blocked: as above |
| figs5d-msn-response-types | dandiset | blocked: no sorted units |
| figs5ef-saline-no-change | dandiset | blocked: no sorted units |
| fig6fg-response-types-shift | dandiset | blocked: no sorted units |
| fig6hi-increase-type-stronger | dandiset | blocked: no sorted units |
| mua-ppx-park-vs-ctrl | dandiset | tested (multi-unit activity) |
| mua-saline-no-change | dandiset | tested; blocked if too few saline sessions cover the windows |
| mua-response-shift-first-fourth | dandiset | tested (multi-unit activity) |
| fig1b-th-depletion | Zenodo | tested |
| fig1c-rotarod | Zenodo | tested |
| fig1fg-baseline-similar | Zenodo | tested |
| fig1f-ctrl-ppx-no-change | Zenodo | tested |
| fig1g-park-ppx-impulsive | Zenodo | tested |
| fig1ik-a-k-auc | Zenodo | tested |
| figs2ik-washout | Zenodo | tested |
| fig2d-ctrl-a77636-no-change | Zenodo | tested |
| fig2e-park-a77636-impulsive | Zenodo | tested |
| fig2fh-a77636-a-k-auc | Zenodo | tested |
| fig2bc-a77636-locomotion | Zenodo | blocked: injection time not in the files |
| fig3a-a-correlates-dms | Zenodo | tested |
| fig3b-k-no-correlation | Zenodo | tested |
| fig5g-mcherry-no-change | Zenodo | tested |
| fig5hi-dreadd-impulsive | Zenodo | tested |
| fig5jl-dreadd-a-k-auc | Zenodo | tested |
| fig5cf-dreadd-locomotion | Zenodo | blocked: injection time and Cre line not in the files |
| fig6bc-ppx-progression | Zenodo | tested |
| fig6de-a77636-no-progression | Zenodo | tested |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats

import icd

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
print("dandiset", icd.DANDISET_ID, "version", icd.DANDISET_VERSION)
T_START = time.time()

## Loading

Metadata of every NWB file, then the threshold crossings of each channel, reduced as they load
to 1-min counts around the injection (`icd.session_activity`). Then the Zenodo deposit.

In [ ]:
sessions = icd.sessions()
meta = pd.DataFrame([{k: s[k] for k in ("path", "subject", "group", "genotype", "genotype_raw", "sex",
                                         "lesion", "drug", "day", "opto_only", "start_time", "n_units")}
                     | {"n_segments": len(s["segments"]),
                        "segments_min": [(round(a, 1), round(l, 1)) for a, l in s["segments"]],
                        "end_min": max(a + l for a, l in s["segments"]),
                        "n_event_channels": len(s["events"]),
                        "session": s["session"]} for s in sessions])
by_path = {s["path"]: s for s in sessions}
t = time.time()
activity = {s["path"]: icd.session_activity(s) for s in sessions if s["events"]}
print(f"streamed threshold crossings of {len(activity)} files in {time.time() - t:.0f} s")
units = {s["path"]: icd.unit_times(s) for s in sessions if s["n_units"]}
read_ok = {p: icd.check_event_read(by_path[p], sorted(by_path[p]["events"])[0])
           for p in list(activity)[:3]}
print("byte-range reads equal h5py reads:", read_ok)

In [ ]:
cog = icd.cognitive_files()
groups = icd.readme_groups()
beh = {name: icd.session_table(files) for name, files in cog.items()}
rot = icd.rotarod()
hist = icd.histology_rois(icd.histology_sheets())
with icd.DandiAPIClient() as client:
    draft = {a.path: a.size for a in client.get_dandiset(icd.DANDISET_ID, "draft").get_assets()}
    pinned = {a.path: a.size for a in client.get_dandiset(icd.DANDISET_ID, icd.DANDISET_VERSION).get_assets()}
print(f"loaded in {time.time() - T_START:.0f} s")

## Data check

What was loaded, and each check of the metadata against the data.

In [ ]:
print("== Versions")
print(f"pinned {icd.DANDISET_VERSION}: {len(pinned)} assets, {sum(pinned.values()) / 1e12:.3f} TB; "
      f"draft: {len(draft)} assets; same paths and sizes: {pinned == draft}")

print("\n== Subjects (from the NWB subject metadata)")
subj = meta.drop_duplicates("subject")[["subject", "group", "lesion", "genotype", "genotype_raw", "sex"]]
print(subj.sort_values(["group", "genotype", "subject"]).to_string(index=False))
print("\nsubjects by group and genotype:")
print(pd.crosstab(subj.group, subj.genotype, margins=True))
print("genotype spellings:", sorted(meta.genotype_raw.unique()))

print("\n== Sessions")
print(meta[["session", "subject", "group", "genotype", "drug", "day", "start_time", "n_segments",
            "end_min", "n_event_channels", "n_units"]].sort_values(["subject", "day"]).to_string(index=False))
ppx = meta[(meta.drug == "PPX") & ~meta.opto_only]
print("\nPPX sessions by group and day (animals):")
print(pd.crosstab(ppx.group, ppx.day, values=ppx.subject, aggfunc="nunique"))
print("saline sessions:", meta[meta.drug == "saline"].subject.tolist())
print("paper: S5D Ctrl N = 7, Park N = 11; Fig 6F/G day 1 Ctrl N = 7, Park N = 10; day 4 Ctrl N = 3, Park N = 8;"
      " Fig 4B/C D1 Ctrl N = 3, Park N = 5, A2a Ctrl N = 4, Park N = 4; S5E saline N = 4")
years = meta.start_time.str[:4].astype(int)
print("session start years:", years.value_counts().sort_index().to_dict())
print("sessions starting in 2025 (after the paper was submitted in 2024):")
print(meta.loc[years >= 2025, ["session", "start_time"]].to_string(index=False))
for s in sessions[:1]:
    print("subject description (first file):", repr(s["description"][:120]))
print("electrode locations across files:", sorted({l for s in sessions for l in s["locations"]}))

print("\n== Recording segments (minutes) and the inferred injection time")
seg = pd.DataFrame([{"session": by_path[p]["session"], "group": by_path[p]["group"],
                     "segments": [(round(a, 1), round(l, 1)) for a, l in by_path[p]["segments"]],
                     "t0_rule": a["t0"], "t0_changepoint": a["cp"], "drop_at_t0": a["drop_at_t0"],
                     "best_drop": a["cp_drop"], "end_min": a["end_min"],
                     "post_min_available": a["end_min"] - a["t0"]} for p, a in activity.items()])
print(seg.round(2).to_string(index=False))

print("\n== Threshold crossings (SpikeEvents) against the recording")
ev = pd.DataFrame([{"session": by_path[p]["session"], "channels": a["n_channels"], "events": a["n_events"],
                    "first_min": a["t_first_min"], "last_min": a["t_last_min"], "recording_end_min": a["end_min"],
                    "outside_segments": a["outside_share"], "sorted": a["sorted"]} for p, a in activity.items()])
print(ev.round(4).to_string(index=False))
print("files whose events lie outside their recording:",
      ev.loc[(ev.first_min < -1) | (ev.last_min > ev.recording_end_min + 1), "session"].tolist())
print("files whose events stop more than 10 min before the recording ends:",
      ev.loc[ev.last_min < ev.recording_end_min - 10, "session"].tolist())
print("channels per file:", ev.channels.min(), "to", ev.channels.max(), "of 32")

print("\n== Units tables")
for p, u in units.items():
    n = np.array([len(v) for v in u.values()])
    span = max(v[-1] for v in u.values() if len(v)) / 60
    print(f"{by_path[p]['session']}: {len(u)} units, spikes per unit {n.min()} to {n.max()} "
          f"(median {np.median(n):.0f}), last spike {span:.1f} min; columns are id, spike_times, unit_name only")
    print("   spike counts in id order:", n.tolist())

### What the files hold for the unit-level claims

The paper's Figures 4, 6F-K and S5 use manually sorted single units, classified as putative
MSNs, and identified as dMSNs or iMSNs by their response to laser pulses at the end of each
session. The cell below lists what each file holds that could carry this.

In [ ]:
print("stimulus groups (presentation / templates contents):",
      sorted({str(s["stimulus"]) for s in sessions}))
print("files with an intervals group:", sum(s["intervals"] for s in sessions),
      "| non-empty analysis groups:", sum(bool(s["analysis"]) for s in sessions))
print("processing modules:", sorted({str(s["processing"]) for s in sessions}))
print("files with a units table:", [(by_path[p]["session"], by_path[p]["unit_columns"]) for p in units])
print("files with SpikeEvents (unsorted threshold crossings with waveforms):",
      sum(bool(s["events"]) for s in sessions), "of", len(sessions))
print("opto-only file(s):", meta.loc[meta.opto_only, "session"].tolist())

### The injection time and the shared clock

The files do not record when PPX or saline was injected. `icd.injection_minute` infers it from
the recording segments (see its docstring). PPX acutely lowers striatal firing (Figure 4D-G),
so if the inferred time is right, the summed threshold-crossing rate should drop just after it
and not at arbitrary other times. The contrast below compares the drop at the inferred time with
the same measure at times shifted by 60 to 180 min (a label-shift null), and with the saline
sessions. A session is taken as one whose clock could not be established when the drop at the
inferred time is above -0.2 (log ratio) and the largest drop in the first 120 min lies more than
10 min away (the rule fixed in the claims file).

In [ ]:
rows = []
for p, a in activity.items():
    s = by_path[p]
    if s["opto_only"] or not np.isfinite(a["t0"]):
        continue
    shifted = [icd.acute_drop(a["total_per_min"], a["t0"] + d) for d in (60, 90, 120, 150, 180)]
    rows.append({"session": s["session"], "group": s["group"], "drug": s["drug"], "t0": a["t0"],
                 "changepoint": a["cp"], "drop_at_t0": a["drop_at_t0"],
                 "median_drop_shifted": np.nanmedian(shifted)})
clock = pd.DataFrame(rows)
clock["clock_failed"] = (clock.drop_at_t0 > -0.2) & ((clock.changepoint - clock.t0).abs() > 10)
clock.loc[clock.drop_at_t0.isna(), "clock_failed"] = True
print(clock.round(2).to_string(index=False))
for drug in ("PPX", "saline"):
    c = clock[(clock.drug == drug) & clock.drop_at_t0.notna()]
    print(f"{drug}: drop at inferred time median {c.drop_at_t0.median():.2f} (n = {len(c)}), "
          f"{(c.drop_at_t0 < -0.2).sum()} below -0.2; shifted median {c.median_drop_shifted.median():.2f}, "
          f"{(c.median_drop_shifted < -0.2).sum()} below -0.2")
c = clock[(clock.drug == "PPX") & clock.drop_at_t0.notna() & clock.median_drop_shifted.notna()]
print("PPX, inferred against shifted, Wilcoxon signed-rank p =",
      f"{stats.wilcoxon(c.drop_at_t0, c.median_drop_shifted).pvalue:.2g}")
print("sessions whose clock could not be established:", clock.loc[clock.clock_failed, "session"].tolist())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for drug, axi in zip(("PPX", "saline"), ax):
    for p, a in activity.items():
        s = by_path[p]
        if s["drug"] != drug or s["opto_only"] or not np.isfinite(a["t0"]):
            continue
        tot = a["total_per_min"]
        i0 = int(a["t0"])
        if i0 < 30 or tot[i0 - 30:i0].mean() == 0:
            continue
        rel = np.arange(len(tot)) - a["t0"]
        axi.plot(rel, tot / tot[i0 - 30:i0].mean(), lw=0.5, color="C1" if s["group"] == "Park" else "C0")
    axi.axvline(0, color="k", lw=0.8)
    axi.set_xlim(-60, 360)
    axi.set_ylim(0, 3)
    axi.set_title(f"{drug} sessions (blue Ctrl, orange Park)")
    axi.set_xlabel("min from inferred injection")
    axi.set_ylabel("summed rate / baseline")
plt.tight_layout()
plt.show()
print("y axis clipped at 3: the opto segments at the end of sessions reach higher ratios")

### The Zenodo deposit

In [ ]:
print("== Delay-discounting sessions (Data_ICD_Fig1&2&5&6_CognitiveAssay.zip)")
for exp, tab in beh.items():
    files = tab.drop_duplicates("file")
    print(f"{exp}: {len(files)} session files of {files.animal.nunique()} animals;",
          files.condition.value_counts().to_dict())
    for g, ids in groups[exp].items():
        have = files[files.animal.isin(ids)]
        counts = have.groupby("animal").condition.value_counts().unstack(fill_value=0)
        missing = sorted(set(ids) - set(have.animal))
        print(f"   {g}: README lists {len(ids)}; with files {have.animal.nunique()}; missing {missing}; "
              f"sessions per animal {counts.min().to_dict()} to {counts.max().to_dict()}")
    extra = sorted(set(files.animal) - {i for ids in groups[exp].values() for i in ids})
    print("   animals with files but in no README group:", extra)
    blocks = tab.groupby("file").present.sum()
    print("   free-choice blocks per session:", blocks.value_counts().to_dict(),
          "| trials per free block:", tab[tab.present].trials.value_counts().to_dict())
    lab = tab[tab.labelled_delay.notna()]
    if len(lab):
        print("   sessions whose block labels state a delay:", lab.file.nunique(), "of animals",
              sorted(lab.animal.unique()), "; labelled delays of blocks B3, B6, B9, B12:",
              lab.groupby("delay").labelled_delay.unique().apply(list).to_dict())
    print("   files without a date in the name:", files[files.date.isna()].file.tolist())
    three = blocks[blocks == 3].index
    if len(three):
        print("   sessions with three free-choice blocks:", len(three), "of animals",
              sorted(tab[tab.file.isin(three)].animal.unique()))
print("paper: PPX Ctrl 16, Park 31; A77636 Ctrl 10, Park 16; mCherry 15, hM4Di 12, hM3Dq 12")

print("\n== Rotarod (Data_ICD_Fig1_Rotarod_PPX.xlsx)")
print(rot.groupby("column").latency_s.agg(["count", "min", "median", "max"]))
print("paper: Ctrl n = 31, Park/PPX n = 45, Park/Saline n = 27")

print("\n== Histology (Data_ICD_Fig3_HistologyQuantification_striatum_raw.xlsx)")
print("ROI rows:", hist.roi.value_counts().to_dict())
print(hist.groupby(["cohort", "group"]).animal.nunique().unstack())
chk = hist.assign(ratio=hist.rawintden / (hist.area * hist["mean"]))
print("RawIntDen / (Area x Mean): median", round(chk.ratio.median(), 4), "range",
      round(chk.ratio.min(), 4), "to", round(chk.ratio.max(), 4))
print("parkinsonian animals:", hist[hist.group == "Park"].animal.nunique(), "| control animals:",
      hist[hist.group == "Ctrl"].animal.nunique(), "(paper Fig 1B n = 44)")

## Multi-unit classification

Each channel of each session is classified by the paper's rule (`icd.classify`). Channels
with a baseline rate under 0.5 Hz are left out, as are sessions whose clock could not be
established and windows that are not fully recorded.

In [ ]:
def classify_all(key="rule", paired=True, alpha=icd.ALPHA):
    out = []
    for p, a in activity.items():
        s = by_path[p]
        if s["opto_only"] or "counts_" + key not in a:
            continue
        counts, cov = a["counts_" + key], a["covered_" + key]
        for ch in counts.columns:
            kind, rb, rp, ratio = icd.classify(counts[ch], cov, alpha=alpha, paired=paired)
            out.append({"path": p, "session": s["session"], "subject": s["subject"], "group": s["group"],
                        "genotype": s["genotype"], "drug": s["drug"], "day": s["day"], "channel": ch,
                        "type": kind, "base_hz": rb, "post_hz": rp, "ratio": ratio})
    out = pd.DataFrame(out)
    failed = set(clock.loc[clock.clock_failed, "session"])
    out["clock_ok"] = ~out.session.isin(failed)
    return out


chan = classify_all()
TYPES = ["increase", "decrease", "no change"]
print("channels classified:", len(chan), "| windows not fully recorded:", (chan.type == "missing").sum(),
      "in sessions", sorted(chan.loc[chan.type == "missing", "session"].unique()))
print("below 0.5 Hz at baseline:", (chan.base_hz < icd.MIN_BASELINE_HZ).sum(),
      "| in sessions whose clock failed:", (~chan.clock_ok).sum())


def usable(df, min_hz=icd.MIN_BASELINE_HZ):
    return df[(df.type != "missing") & (df.base_hz >= min_hz) & df.clock_ok]


def type_table(df, by):
    return pd.crosstab(df[by], df.type).reindex(columns=TYPES, fill_value=0)


def shares(tab):
    return (tab.T / tab.sum(axis=1)).T

## Claim mua-ppx-park-vs-ctrl

> PPX at 4-5 h changes striatal firing in parkinsonian mice in both directions and in a larger
> share of recording sites than in control mice, tested on the multi-unit activity of each
> channel (the dandiset's counterpart of Supplementary Figure 5D).

In [ ]:
rng_mua = np.random.default_rng(1)
ppx_chan = usable(chan[chan.drug == "PPX"])
first_day = ppx_chan.groupby("subject").day.min()
readings = {
    "a: all PPX sessions": ppx_chan,
    "b: first available session per animal": ppx_chan[ppx_chan.day == ppx_chan.subject.map(first_day)],
    "c: day-1 sessions": ppx_chan[ppx_chan.day == 1],
}
verdicts = {}
for name, df in readings.items():
    tab = type_table(df, "group")
    sh = shares(tab)
    p = icd.fisher_freeman_halton(tab.to_numpy(), 20000, rng_mua)
    changed = 1 - sh["no change"]
    print(f"--- reading {name}: {df.subject.nunique()} animals, {df.session.nunique()} sessions, {len(df)} channels")
    print(pd.concat([tab, sh.round(3).add_suffix(" share")], axis=1).to_string())
    print(f"changed share Park {changed['Park']:.3f}, Ctrl {changed['Ctrl']:.3f}; Freeman-Halton p = {p:.4f}")
    higher = changed["Park"] > changed["Ctrl"]
    both = sh.loc["Park", "increase"] >= 0.10 and sh.loc["Park", "decrease"] >= 0.10
    one = (sh.loc["Park", "increase"] >= 0.10) or (sh.loc["Park", "decrease"] >= 0.10)
    if higher and p < 0.05 and both:
        v = "reproduced"
    elif higher and both:
        v = "partially_reproduced"
    elif higher and p < 0.05 and one:
        v = "partially_reproduced"
    else:
        v = "not_reproduced"
    verdicts[name] = v
    print("criterion:", v)
print("\nverdicts by reading:", verdicts)

In [ ]:
df = readings["a: all PPX sessions"]
per_animal = shares(type_table(df, "subject")).join(df.groupby("subject")[["group", "genotype"]].first())
per_animal["sessions"] = df.groupby("subject").session.nunique()
per_animal["channels"] = df.groupby("subject").size()
print(per_animal.round(3).sort_values(["group", "genotype"]).to_string())
for col in TYPES:
    a_, b_ = per_animal.loc[per_animal.group == "Park", col], per_animal.loc[per_animal.group == "Ctrl", col]
    print(f"{col}: per-animal median Park {a_.median():.3f} (n = {len(a_)}), Ctrl {b_.median():.3f} (n = {len(b_)}),"
          f" Mann-Whitney p = {stats.mannwhitneyu(a_, b_).pvalue:.3f}")
per_session = shares(type_table(df, "session")).join(df.groupby("session")[["group", "subject", "day"]].first())
print("\nper session:")
print(per_session.round(2).sort_values(["group", "subject", "day"]).to_string())
print("\nrate ratio (240-300 min over baseline) per channel: median Park "
      f"{df[df.group == 'Park'].ratio.median():.3f}, Ctrl {df[df.group == 'Ctrl'].ratio.median():.3f};"
      f" range {df.ratio.min():.2f} to {df.ratio.max():.2f}")

In [ ]:
print("Variants on reading a (all PPX sessions):")
var_rows = []
for label, d in [
        ("main: paired, 0.5 Hz, rule clock", usable(chan[chan.drug == "PPX"])),
        ("threshold 0 Hz (any crossings)", usable(chan[chan.drug == "PPX"], 1e-9)),
        ("threshold 2 Hz", usable(chan[chan.drug == "PPX"], 2.0)),
        ("rank-sum test", usable(classify_all(paired=False).query("drug == 'PPX'"))),
        ("change-point clock, all sessions", usable(classify_all(key="cp").assign(clock_ok=True).query("drug == 'PPX'"))),
        ("rule clock, including clock-failed sessions", usable(chan[chan.drug == "PPX"].assign(clock_ok=True)))]:
    tab = type_table(d, "group")
    sh = shares(tab)
    p = icd.fisher_freeman_halton(tab.to_numpy(), 5000, rng_mua)
    var_rows.append({"variant": label, "channels": len(d),
                     **{f"Park {k}": sh.loc["Park", k] for k in TYPES},
                     **{f"Ctrl {k}": sh.loc["Ctrl", k] for k in TYPES}, "p (5000 perm.)": p})
print(pd.DataFrame(var_rows).round(3).to_string(index=False))

print("\nUnits tables (two files; no cell types, all units classified, baseline >= 0.5 Hz):")
for p, u in units.items():
    s = by_path[p]
    t0 = icd.injection_minute(s["segments"])
    counts = icd.minute_counts(u, t0)
    cov = icd.covered_minutes(s["segments"], t0, counts.index)
    res = pd.DataFrame([icd.classify(counts[k], cov) for k in counts.columns],
                       columns=["type", "base_hz", "post_hz", "ratio"], index=counts.columns)
    keep = res[res.base_hz >= icd.MIN_BASELINE_HZ]
    print(f"{s['session']} ({s['group']}, {s['genotype']}, day {s['day']}), injection at {t0:.1f} min: "
          f"{len(res)} units, {len(keep)} at >= 0.5 Hz:", keep.type.value_counts().to_dict(),
          "| all units:", res.type.value_counts().to_dict())

fig, ax = plt.subplots(figsize=(5, 3))
sh = shares(type_table(readings["a: all PPX sessions"], "group"))
sh[TYPES].plot.bar(stacked=True, ax=ax, color=["C3", "C0", "0.7"])
ax.set_ylabel("share of channels")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

**Result: partially reproduced, under reading (a) only.** Pooling all PPX sessions (16 animals, 41 sessions, 1032 channels), 82.2% of parkinsonian channels change at 4-5 h against 77.0% of control channels (Freeman-Halton p = 0.0007), but the change is almost all in one direction: 76.4% decrease and 5.9% increase in parkinsonian mice, so the increase type stays under the 10% the criterion asks for. Under readings (b) and (c) the changed share is not higher in parkinsonian mice (88.6% against 89.7%, and 88.5% against 96.0%). Per animal the shares do not differ (increase median 0.016 against 0.010, Mann-Whitney p = 0.561). Most channels in both groups decrease, and in the one saline session that covers the windows 11 of 16 channels also decrease (next section), so the multi-unit rate falls over these long sessions whatever the drug; this test cannot separate dMSNs from iMSNs, which the paper's claim is about. The variants (thresholds, rank-sum test, change-point clock) give the same picture, with parkinsonian increase shares of 5.6% to 6.1%. In the two units tables, 6 of 41 units in E0 day 4 (parkinsonian) increase and 26 decrease; 2 of 29 in P4 day 1 (control) increase and 12 decrease.

## Claim mua-saline-no-change

> After a saline injection nearly all recording sites of parkinsonian mice show no change in
> firing, tested on the multi-unit activity of each channel (counterpart of Supplementary
> Figure 5F).

In [ ]:
sal = chan[chan.drug == "saline"]
cover = sal.groupby("session").apply(lambda d: (d.type != "missing").any(), include_groups=False)
print("saline sessions and whether both post-injection windows are recorded (injection at 30 min):")
print(seg[seg.session.isin(sal.session.unique())][["session", "end_min", "t0_rule", "post_min_available"]]
      .assign(covers_both=lambda d: d.session.map(cover)).round(1).to_string(index=False))
n_cover = int(cover.sum())
print(f"{n_cover} of {len(cover)} saline sessions cover both windows; the criterion needs three")
ok = usable(sal.assign(clock_ok=True))
print("classified channels:", type_table(ok, "session").to_string())
if len(ok):
    print(f"no-change share: {(ok.type == 'no change').mean():.3f} of {len(ok)} channels")
# variant: the last 60 recorded minutes, in two 30-min halves, for every saline session
rows = []
for p, a in activity.items():
    s = by_path[p]
    if s["drug"] != "saline":
        continue
    post_end = int(np.floor(a["end_min"] - a["t0"])) - 1
    post = [(post_end - 59, post_end - 30), (post_end - 29, post_end)]
    for ch in a["counts_rule"].columns:
        kind, rb, rp, ratio = icd.classify(a["counts_rule"][ch], a["covered_rule"], post=post)
        rows.append({"session": s["session"], "post_windows": post, "type": kind, "base_hz": rb, "ratio": ratio})
late = pd.DataFrame(rows)
late = late[(late.type != "missing") & (late.base_hz >= icd.MIN_BASELINE_HZ)]
print("\nvariant, last 60 recorded minutes of each saline session:")
print(type_table(late, "session").join(late.groupby("session").post_windows.first()).to_string())
print(f"no-change share {(late.type == 'no change').mean():.3f} of {len(late)} channels")
lp = usable(chan[chan.drug == "PPX"])
print("for comparison, PPX sessions of parkinsonian mice (main analysis):",
      f"no-change share {(lp[lp.group == 'Park'].type == 'no change').mean():.3f}")

**Result: blocked (too few sessions).** Only 1 of the 4 saline sessions (H1-3) is recorded until 300 min after the inferred injection, where the criterion needs three; the others end 251 to 281 min after it. In H1-3, 4 of 16 channels (25.0%) show no change. As a variant over the last 60 recorded minutes of each session, 48.2% of 83 channels show no change, against 17.8% in PPX sessions of parkinsonian mice. The saline recordings in the dandiset are shorter than the paper's 330-min sessions, or their injection was earlier than the description says; the files do not tell which.

## Claim mua-response-shift-first-fourth

> In parkinsonian mice the response types of recording sites shift between the first and the
> fourth PPX session towards more increases and fewer decreases, while in control mice they do
> not, tested on the multi-unit activity of each channel (counterpart of Figure 6F and G).

In [ ]:
rng_shift = np.random.default_rng(3)
d14 = usable(chan[(chan.drug == "PPX") & chan.day.isin([1, 4])])
res = {}
for grp in ("Park", "Ctrl"):
    d = d14[d14.group == grp]
    tab = type_table(d, "day")
    sh = shares(tab)
    p = icd.fisher_freeman_halton(tab.to_numpy(), 20000, rng_shift) if tab.shape[0] == 2 else np.nan
    res[grp] = (sh, p)
    print(f"--- {grp}: animals day 1 {sorted(d[d.day == 1].subject.unique())}, day 4 {sorted(d[d.day == 4].subject.unique())}")
    print(pd.concat([tab, sh.round(3).add_suffix(" share")], axis=1).to_string())
    print(f"Freeman-Halton p = {p:.4f}")
sh, p = res["Park"]
inc_up = sh.loc[4, "increase"] > sh.loc[1, "increase"]
dec_down = sh.loc[4, "decrease"] < sh.loc[1, "decrease"]
ctrl_p = res["Ctrl"][1]
if p < 0.05 and inc_up and dec_down and ctrl_p >= 0.05:
    v = "reproduced"
elif p < 0.05 and (inc_up and dec_down) and ctrl_p < 0.05:
    v = "partially_reproduced"
elif p < 0.05 and (inc_up or dec_down):
    v = "partially_reproduced"
else:
    v = "not_reproduced"
print(f"\nPark p = {p:.4f}, increase up {inc_up}, decrease down {dec_down}; Ctrl p = {ctrl_p:.4f}; criterion: {v}")

In [ ]:
both_days = d14.groupby("subject").day.nunique()
paired_animals = both_days[both_days == 2].index
d = d14[d14.subject.isin(paired_animals)]
print("variant, animals with both day 1 and day 4:", sorted(paired_animals))
for grp in ("Park", "Ctrl"):
    dd = d[d.group == grp]
    if dd.empty:
        print(grp, "none")
        continue
    tab = type_table(dd, "day")
    print(grp, "\n", pd.concat([tab, shares(tab).round(3).add_suffix(" share")], axis=1).to_string(),
          f"\n p = {icd.fisher_freeman_halton(tab.to_numpy(), 5000, rng_shift):.4f}" if tab.shape[0] == 2 else "")
print("\nper animal and day (Park):")
print(shares(type_table(d14[d14.group == "Park"].assign(key=lambda x: x.subject + " day " + x.day.astype(str)), "key")).round(2).to_string())
inc = d14[(d14.group == "Park") & (d14.type == "increase")]
print("\nincrease-type channels, Park: day 1 n =", (inc.day == 1).sum(), "day 4 n =", (inc.day == 4).sum())
if (inc.day == 1).sum() and (inc.day == 4).sum():
    print(f"rate ratio day 1 median {inc[inc.day == 1].ratio.median():.2f}, day 4 {inc[inc.day == 4].ratio.median():.2f},"
          f" Mann-Whitney p = {stats.mannwhitneyu(inc[inc.day == 1].ratio, inc[inc.day == 4].ratio).pvalue:.3f}")

**Result: partially reproduced.** In parkinsonian mice the types shift from day 1 to day 4 as the paper describes: increase 0.9% to 18.0%, decrease 87.6% to 59.4% (Freeman-Halton p < 0.0001; 9 animals on day 1, 5 on day 4). In control mice the types also shift (decrease 96.0% to 57.4%, no change 4.0% to 42.6%, p < 0.0001), where the paper finds no shift; no control channel increases on either day. Restricted to animals with both sessions the result is the same (parkinsonian increase 0% to 18.3%, p = 0.0002; control p = 0.0004). The day-4 increases come mainly from J0 (61% of its channels) and K1 (17%). The rate ratio of increase-type channels is not higher on day 4 (median 2.30 on day 1, n = 2, against 1.63, n = 24).

## Behaviour: shared steps

Each animal's curve in a condition is the mean over its sessions of the percentage of
large-reward choices in each free-choice block (`icd.animal_curves`). Comparisons at each
delay are paired t-tests with p x 4 (`icd.paired_by_delay`); the two-way repeated-measures
ANOVA is `icd.rm_anova_2x4`. Animals without a complete curve in both conditions are left out
and listed.

In [ ]:
P_ANIMALS = sorted(a for a in beh["PPX"].animal.unique() if a.startswith("P"))


def paired_curves(exp, animals, a="baseline", b="drug", orders_a=None, orders_b=None):
    ca = icd.animal_curves(beh[exp], animals, a, orders_a)
    cb = icd.animal_curves(beh[exp], animals, b, orders_b)
    keep = ca.dropna().index.intersection(cb.dropna().index)
    dropped = sorted(set(animals) - set(keep))
    return ca.loc[keep], cb.loc[keep], dropped


def report(exp, animals, a="baseline", b="drug", orders_a=None, orders_b=None, label=""):
    ca, cb, dropped = paired_curves(exp, animals, a, b, orders_a, orders_b)
    print(f"{label}: {len(ca)} of {len(animals)} animals with complete curves; left out {dropped}")
    an = icd.rm_anova_2x4(ca, cb)
    print("  RM ANOVA:", {k: f"F({v[1]},{v[2]}) = {v[0]:.2f}, p = {v[3]:.2g}" for k, v in an.items()})
    tab = icd.paired_by_delay(ca, cb)
    print(tab.round(4).to_string(index=False))
    return ca, cb, tab, an

## Claim fig1fg-baseline-similar

> PPX-naive control and parkinsonian mice show similar delay discounting (p > 0.05 at all
> delays).

In [ ]:
ctrl_ids, park_ids = groups["PPX"]["Control"], groups["PPX"]["Parkinsonian"]
cb_ctrl = icd.animal_curves(beh["PPX"], ctrl_ids, "baseline").dropna()
cb_park = icd.animal_curves(beh["PPX"], park_ids, "baseline").dropna()


def between(c1, c2):
    rows = []
    for d in icd.DELAYS:
        t = stats.ttest_ind(c1[d], c2[d])
        rows.append({"delay_s": d, "mean_ctrl": c1[d].mean(), "mean_park": c2[d].mean(),
                     "diff": c2[d].mean() - c1[d].mean(), "p_bonferroni": min(1, 4 * t.pvalue),
                     "p_mannwhitney_x4": min(1, 4 * stats.mannwhitneyu(c1[d], c2[d]).pvalue)})
    return pd.DataFrame(rows)


print(f"Ctrl {len(cb_ctrl)} of {len(ctrl_ids)} with a complete baseline curve; Park {len(cb_park)} of {len(park_ids)}")
bs = between(cb_ctrl, cb_park)
print(bs.round(4).to_string(index=False))
print("any delay with p < 0.05:", bool((bs.p_bonferroni < 0.05).any()))
c_noP = cb_ctrl.drop(index=[a for a in P_ANIMALS if a in cb_ctrl.index])
print(f"\nvariant without the P animals (Ctrl n = {len(c_noP)}):")
print(between(c_noP, cb_park).round(4).to_string(index=False))

**Result: reproduced.** No delay differs between PPX-naive control (14 of 16 with a complete curve) and parkinsonian mice (31): differences of -5.0 to 1.6 points, all adjusted p of 0.92 or more. Without the six P animals the result is the same (all adjusted p = 1).

## Claim fig1f-ctrl-ppx-no-change

> PPX does not change delay discounting in healthy control mice (n = 16, p > 0.05 at all
> delays).

In [ ]:
c_a, c_b, c_tab, c_an = report("PPX", ctrl_ids, label="Ctrl, baseline against PPX")
print("any delay with adjusted p < 0.05:", bool((c_tab.p_bonferroni < 0.05).any()))
print("\nvariant without the P animals:")
_ = report("PPX", [a for a in ctrl_ids if a not in P_ANIMALS], label="Ctrl without P animals")
print("\nthe P animals, with their first three blocks (labelled 0, 6 and 12 s):")
pa = icd.animal_curves(beh["PPX"], P_ANIMALS, "baseline")
pb = icd.animal_curves(beh["PPX"], P_ANIMALS, "drug")
print(pd.concat({"baseline": pa, "PPX": pb}, axis=1).round(1).to_string())

**Result: reproduced, on 10 of the 16 control mice.** PPX lowers the choice by 1.6 to 8.4 points, with no delay significant (adjusted p 0.23 to 1; condition F(1,9) = 2.52, p = 0.15). The six P animals are left out because none of their PPX sessions has a fourth free-choice block; their sessions label the blocks as delays of 0, 6, 12 and 18 s, not 0, 3, 6 and 9 s.

## Claim fig1g-park-ppx-impulsive

> In parkinsonian mice PPX reduces the choice of the delayed large reward at every delay
> (n = 31; 0, 3 and 6 s p < 0.001, 9 s p = 0.0087).

In [ ]:
p_a, p_b, p_tab, p_an = report("PPX", park_ids, label="Park, baseline against PPX")
lower = (p_tab["diff"] < 0) & (p_tab.p_bonferroni < 0.05)
print("delays lower on PPX with adjusted p < 0.05:", int(lower.sum()), "of 4")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for axi, (ca, cb, title) in zip(ax, [(c_a, c_b, "Ctrl"), (p_a, p_b, "Park")]):
    axi.errorbar(icd.DELAYS, ca.mean(), ca.sem(), marker="o", mfc="white", label="baseline")
    axi.errorbar(icd.DELAYS, cb.mean(), cb.sem(), marker="o", label="PPX")
    axi.set_title(f"{title} (n = {len(ca)})")
    axi.set_xlabel("delay (s)")
ax[0].set_ylabel("% large-reward choices")
ax[0].set_ylim(0, 100)
ax[1].legend()
plt.tight_layout()
plt.show()

**Result: reproduced.** In 30 of 31 parkinsonian mice (E6 has no complete PPX curve), PPX lowers the large-reward choice at all four delays: by 12.5, 14.7, 9.7 and 6.4 points at 0, 3, 6 and 9 s (adjusted p = 0.0007, < 0.0001, < 0.0001 and 0.0034; condition F(1,29) = 48.08, p = 1.3e-07). The Wilcoxon variant gives the same.

## Claim fig1ik-a-k-auc

> In parkinsonian mice PPX decreases A, increases K and reduces the area under the discounting
> curve (n = 31; p < 0.0001, p = 0.03, p < 0.0001).

In [ ]:
def fits_for(exp, animals, conds):
    out = {}
    for c in conds:
        out[c] = icd.fit_table(icd.animal_curves(beh[exp], animals, c).dropna())
    return out


def friedman_report(fits, conds, pairs):
    res = {}
    for m in ("A", "K", "AUC"):
        tab = pd.concat({c: fits[c][m] for c in conds}, axis=1)
        fr, dunn = icd.friedman_dunn(tab, pairs)
        res[m] = dunn
        print(f"{m}: n complete = {len(tab.dropna())}, medians {tab.dropna().median().round(3).to_dict()}, "
              f"Friedman p = {fr:.2g}; Dunn " + "; ".join(f"{a} vs {b}: z = {z:.2f}, p = {p:.2g}"
                                                          for (a, b), (z, p) in dunn.items()))
    return res


park_fits = fits_for("PPX", park_ids, ["baseline", "drug", "washout"])
for c, f in park_fits.items():
    print(f"{c}: {len(f)} animals fitted, failed fits {int(f.A.isna().sum())}; A range {f.A.min():.2f} to {f.A.max():.2f},"
          f" K range {f.K.min():.3f} to {f.K.max():.3f}")
fr_park = friedman_report(park_fits, ["baseline", "drug", "washout"], [("baseline", "drug"), ("baseline", "washout")])
ok_A = fr_park["A"][("baseline", "drug")][0] < 0 and fr_park["A"][("baseline", "drug")][1] < 0.05
ok_K = fr_park["K"][("baseline", "drug")][0] > 0 and fr_park["K"][("baseline", "drug")][1] < 0.05
ok_AUC = fr_park["AUC"][("baseline", "drug")][0] < 0 and fr_park["AUC"][("baseline", "drug")][1] < 0.05
print(f"A lower: {ok_A}; K higher: {ok_K}; AUC lower: {ok_AUC}")
both = park_fits["baseline"].index.intersection(park_fits["drug"].index)
for m in ("A", "K", "AUC"):
    print(f"variant, Wilcoxon signed-rank baseline vs PPX, {m}: p = "
          f"{stats.wilcoxon(park_fits['baseline'].loc[both, m], park_fits['drug'].loc[both, m]).pvalue:.2g} (n = {len(both)})")

**Result: partially reproduced.** In 30 parkinsonian mice PPX lowers A (median 0.891 to 0.785, Dunn's p = 1.9e-06) and the AUC (3.948 to 2.905, p = 2.7e-08). K rises (0.258 to 0.321) but Dunn's test gives p = 0.19; the Friedman test over the three conditions gives p = 0.032, close to the paper's 0.03, and a Wilcoxon test of baseline against PPX gives p = 0.045.

## Claim figs2ik-washout

> After a 48 h washout A returns to baseline while the changes in K and AUC persist (baseline
> against washout, A p > 0.99, K p = 0.01, AUC p < 0.01).

In [ ]:
for m in ("A", "K", "AUC"):
    z, p = fr_park[m][("baseline", "washout")]
    print(f"{m}: baseline vs washout z = {z:.2f}, p = {p:.3g}")
w_A = fr_park["A"][("baseline", "washout")][1] >= 0.05
w_K = fr_park["K"][("baseline", "washout")][0] > 0 and fr_park["K"][("baseline", "washout")][1] < 0.05
w_AUC = fr_park["AUC"][("baseline", "washout")][0] < 0 and fr_park["AUC"][("baseline", "washout")][1] < 0.05
print(f"A not different: {w_A}; K higher: {w_K}; AUC lower: {w_AUC}")

**Result: partially reproduced.** After washout A is back at baseline (Dunn's p = 0.73) and K is higher than at baseline (0.419 against 0.258, p = 0.020), but the AUC is not significantly lower (3.379 against 3.948, p = 0.078).

## Claim fig2d-ctrl-a77636-no-change

> The D1 agonist A77636 does not change delay discounting in healthy mice (n = 10, p > 0.05 at
> all delays).

In [ ]:
a_ctrl, a_park = groups["A77636"]["Control"], groups["A77636"]["Parkinsonian"]
ac_a, ac_b, ac_tab, _ = report("A77636", a_ctrl, label="Ctrl, baseline against A77636")
print("any delay with adjusted p < 0.05:", bool((ac_tab.p_bonferroni < 0.05).any()))

**Result: reproduced.** In all 10 control mice A77636 changes the choice by -4.2 to 3.1 points, with no delay significant (adjusted p 0.56 to 1; condition p = 0.67).

## Claim fig2e-park-a77636-impulsive

> In parkinsonian mice A77636 reduces the choice of the delayed large reward (n = 16; 0 s p =
> 0.57, 3 s p = 0.0047, 6 s p = 0.0003, 9 s p = 0.0002).

In [ ]:
ap_a, ap_b, ap_tab, _ = report("A77636", a_park, label="Park, baseline against A77636")
sig = ap_tab[(ap_tab.delay_s > 0) & (ap_tab["diff"] < 0) & (ap_tab.p_bonferroni < 0.05)]
print("of 3, 6 and 9 s, lower with adjusted p < 0.05:", sig.delay_s.tolist())

**Result: reproduced.** In all 16 parkinsonian mice A77636 lowers the choice at 3, 6 and 9 s by 15.3, 19.0 and 19.3 points (adjusted p = 0.0015, 0.0024 and 0.0053), and not at 0 s (6.7 points, p = 0.83), the pattern the paper reports.

## Claim fig2fh-a77636-a-k-auc

> In parkinsonian mice A77636 has no significant effect on A (p = 0.31) but increases K
> (p = 0.0053) and reduces the AUC (p = 0.0008).

In [ ]:
a_fits = fits_for("A77636", a_park, ["baseline", "drug", "washout"])
for c, f in a_fits.items():
    print(f"{c}: {len(f)} animals fitted, failed fits {int(f.A.isna().sum())}")
fr_a = friedman_report(a_fits, ["baseline", "drug", "washout"], [("baseline", "drug"), ("baseline", "washout")])
k_ok = fr_a["K"][("baseline", "drug")][0] > 0 and fr_a["K"][("baseline", "drug")][1] < 0.05
auc_ok = fr_a["AUC"][("baseline", "drug")][0] < 0 and fr_a["AUC"][("baseline", "drug")][1] < 0.05
a_ns = fr_a["A"][("baseline", "drug")][1] >= 0.05
print(f"K higher: {k_ok}; AUC lower: {auc_ok}; A not significant: {a_ns}")

**Result: reproduced.** In 16 parkinsonian mice A77636 raises K (median 0.115 to 0.291, Dunn's p = 0.0053) and lowers the AUC (5.817 to 3.971, p = 0.00081) with no effect on A (p = 0.31). These p-values equal the paper's (0.0053, 0.0008, 0.31).

## Claim fig1b-th-depletion

> The bilateral 6-OHDA lesion leaves about 50% of the TH signal in the rostral DLS and DMS, with
> less depletion in the ventral striatum.

In [ ]:
def residual_th(hist, background, from_mean=False):
    h = hist.copy()
    bg = (h[h.roi.isin(background)].groupby(["cohort", "group", "slice"])["mean"].mean().rename("bg"))
    h = h[h.roi.isin(["DLS", "DMS", "VLS", "VMS"])].join(bg, on=["cohort", "group", "slice"])
    h["ctf"] = (h["mean"] - h.bg) if from_mean else (h.rawintden - h.area * h.bg) / h.area
    h["region"] = h.roi.replace({"VLS": "VS", "VMS": "VS"})
    per = h.groupby(["cohort", "group", "animal", "region"]).ctf.mean().unstack("region")
    ctrl = per.xs("Ctrl", level="group").groupby("cohort").mean()
    park = per.xs("Park", level="group")
    norm = park.div(ctrl.reindex(park.index.get_level_values("cohort")).to_numpy())
    return norm.reset_index().groupby("animal")[["DLS", "DMS", "VS"]].mean()


th = {}
for label, bgset in [("a: background rows", ["background"]), ("b: blank rows", ["blank"]),
                     ("c: all eight", ["background", "blank"])]:
    r = residual_th(hist, bgset)
    th[label] = r
    h, dunn = icd.kruskal_dunn({k: r[k].dropna().to_numpy() for k in ("DLS", "DMS", "VS")},
                               [("DMS", "DLS"), ("VS", "DMS"), ("VS", "DLS")])
    med = r.median()
    print(f"--- reading {label}: {len(r)} parkinsonian animals; median residual TH DLS {med.DLS:.3f}, "
          f"DMS {med.DMS:.3f}, VS {med.VS:.3f} (ranges DLS {r.DLS.min():.2f}-{r.DLS.max():.2f}, "
          f"DMS {r.DMS.min():.2f}-{r.DMS.max():.2f}, VS {r.VS.min():.2f}-{r.VS.max():.2f})")
    print(f"    Kruskal-Wallis p = {h:.2g}; Dunn " + "; ".join(f"{a} vs {b}: z = {z:.2f}, p = {p:.2g}"
                                                         for (a, b), (z, p) in dunn.items()))
    within = abs(med.DLS - 0.5) <= 0.05 and abs(med.DMS - 0.5) <= 0.05
    order = (dunn[("VS", "DMS")][0] > 0 and dunn[("VS", "DMS")][1] < 0.05 and dunn[("VS", "DLS")][0] > 0
             and dunn[("VS", "DLS")][1] < 0.05 and dunn[("DMS", "DLS")][1] >= 0.05)
    print(f"    DLS and DMS within 0.45-0.55: {within}; ordering as in the paper: {order}")
print("\nmeans (the paper reports means):", {k: v.mean().round(3).to_dict() for k, v in th.items()})
bad = hist[(hist.rawintden / (hist.area * hist["mean"]) - 1).abs() > 0.05]
print(f"\nrows where RawIntDen differs from Area x Mean by more than 5%: {len(bad)} of {len(hist)},"
      f" in slices {sorted(bad.slice.unique())}")
print("median RawIntDen / (Area x Mean) in those slices:",
      (bad.rawintden / (bad.area * bad["mean"])).groupby(bad.slice).median().round(3).to_dict())
for label, bgset in [("a", ["background"]), ("b", ["blank"])]:
    r = residual_th(hist, bgset, from_mean=True)
    print(f"variant, corrected fluorescence from Mean - background, reading {label}: medians "
          f"{r.median().round(3).to_dict()}")

**Result: reproduced, under reading (b).** With the blank rows as background, the median residual TH of 45 parkinsonian animals is 0.465 in the DLS and 0.507 in the DMS, both within 0.45-0.55, and 0.677 in the VS; VS is higher than DMS (Dunn's p = 0.00025) and DLS (p = 1.9e-08), and DMS does not differ from DLS (p = 0.18). With the background rows (reading a) the ordering is the same but the values are lower (DLS 0.343, DMS 0.418), outside tolerance, as with all eight rows (0.402, 0.474). The sheet holds 45 parkinsonian animals where the paper gives n = 44.

## Claim fig1c-rotarod

> Parkinsonian mice have a shorter latency to fall on the accelerating rotarod, which PPX
> (0.5 mg/kg) improves and saline does not.

In [ ]:
cols = {c: rot[rot.column == c].latency_s.to_numpy() for c in rot.column.unique()}
print({c: len(v) for c, v in cols.items()})
names = list(cols)
ctrl_c = [c for c in names if c.startswith("Ctrl")][0]
b1, ppx_c = [c for c in names if "cohort1" in c and "baseline" in c][0], [c for c in names if "PPX" in c][0]
b2, sal_c = [c for c in names if "cohort2" in c and "baseline" in c][0], [c for c in names if "Saline" in c][0]
h, dunn = icd.kruskal_dunn(cols, [(ctrl_c, (b1, b2)), (ppx_c, b1), (sal_c, b2)])
print(f"Kruskal-Wallis p = {h:.2g}")
for (a, b), (z, p) in dunn.items():
    print(f"  {a} vs {b}: z = {z:.2f}, p = {p:.3g}")
print("medians:", {c: float(np.median(v)) for c, v in cols.items()})
i_ok = dunn[(ctrl_c, (b1, b2))][0] > 0 and dunn[(ctrl_c, (b1, b2))][1] < 0.05
ii_ok = dunn[(ppx_c, b1)][0] > 0 and dunn[(ppx_c, b1)][1] < 0.05
iii_ok = dunn[(sal_c, b2)][1] >= 0.05
print(f"(i) Ctrl above Park: {i_ok}; (ii) PPX above baseline: {ii_ok}; (iii) saline no change: {iii_ok}")
print("variant, paired Wilcoxon: PPX vs cohort-1 baseline p =",
      f"{stats.wilcoxon(cols[ppx_c], cols[b1]).pvalue:.3g}; saline vs cohort-2 baseline p = "
      f"{stats.wilcoxon(cols[sal_c], cols[b2]).pvalue:.3g}")

**Result: reproduced.** Control mice stay on longer than parkinsonian mice (medians 91.3 against 47.3 and 51.0 s; Dunn's p = 4.7e-16), PPX improves cohort 1 (47.3 to 66.7 s, p = 1.8e-05) and saline does not change cohort 2 (51.0 to 62.3 s, p = 0.83). The paired Wilcoxon variant gives p = 3.1e-08 for PPX and p = 0.074 for saline. The sheet holds 40 control mice where the paper gives 31; the parkinsonian counts (45 and 27) match.

## Claim fig3a-a-correlates-dms

> In parkinsonian mice treated with PPX, A correlates with residual TH in the DMS (r = 0.51,
> p = 0.006, n = 28) but not in the DLS (p = 0.11) or VS (p = 0.07).

In [ ]:
th_a = th["a: background rows"]
fa = park_fits["drug"]
matched = fa.index.intersection(th_a.index)
print(f"PPX parkinsonian animals with a fit on PPX: {len(fa)}; with histology: {len(th_a)}; matched by ID: {len(matched)}")
print("fitted animals without histology:", sorted(set(fa.index) - set(th_a.index)))
print("parkinsonian histology IDs that are not PPX animals:", sorted(set(th_a.index) - set(park_ids)))
corr = {}
for region in ("DLS", "DMS", "VS"):
    d = pd.concat([fa.loc[matched], th_a.loc[matched]], axis=1).dropna(subset=["A", region])
    r, p = stats.pearsonr(d[region], d["A"])
    corr[region] = (r, p, len(d))
    print(f"A vs {region}: r = {r:.3f}, p = {p:.3g}, n = {len(d)}")
for label in ("b: blank rows", "c: all eight"):
    t2 = th[label]
    m2 = fa.index.intersection(t2.index)
    print(f"variant {label}: " + "; ".join(f"{reg} r = {stats.pearsonr(t2.loc[m2, reg], fa.loc[m2, 'A'])[0]:.3f},"
                                          f" p = {stats.pearsonr(t2.loc[m2, reg], fa.loc[m2, 'A'])[1]:.3g}"
                                          for reg in ("DLS", "DMS", "VS")))

**Result: not reproduced.** Of 30 parkinsonian mice with a fit on PPX, 24 have histology under the same ID (the paper has 28). A on PPX does not correlate significantly with residual TH in any region: DMS r = 0.309 (p = 0.14), DLS r = 0.221 (p = 0.30), VS r = 0.268 (p = 0.21). The other background readings give DMS r = 0.255 and 0.281, also not significant. Six fitted animals (C3.1, E3.3, E4, H4.1, N1, N5.1) have no histology under their ID.

## Claim fig3b-k-no-correlation

> K on PPX does not correlate with residual TH in DMS, DLS or VS.

In [ ]:
for region in ("DLS", "DMS", "VS"):
    d = pd.concat([fa.loc[matched], th_a.loc[matched]], axis=1).dropna(subset=["K", region])
    r, p = stats.pearsonr(d[region], d["K"])
    print(f"K vs {region}: r = {r:.3f}, p = {p:.3g}, n = {len(d)}")
    rs = stats.spearmanr(d[region], d["K"])
    print(f"   variant Spearman rho = {rs.statistic:.3f}, p = {rs.pvalue:.3g}")
print(f"K range on PPX: {fa.K.min():.3f} to {fa.K.max():.3f}")

**Result: reproduced.** K on PPX does not correlate with residual TH in any region at p < 0.05 (DLS r = 0.402, p = 0.051; DMS r = 0.039, p = 0.86; VS r = 0.310, p = 0.14; n = 24); the DLS value is close to the threshold. The paper reports DLS p = 0.07, DMS p = 0.92 and VS p = 0.36.

## Claim fig5g-mcherry-no-change

> CNO does not change delay discounting in parkinsonian mice expressing mCherry (n = 15).

In [ ]:
dr = groups["DREADD"]
m_a, m_b, m_tab, _ = report("DREADD", dr["mCherry"], label="mCherry, pre against CNO")
print("any delay with adjusted p < 0.05:", bool((m_tab.p_bonferroni < 0.05).any()))

**Result: reproduced.** In 14 of 15 mCherry mice (R5.5 has no files), CNO changes the choice by -3.8 to 0.4 points, with all adjusted p = 1 (condition p = 0.51).

## Claim fig5hi-dreadd-impulsive

> Chemogenetic inhibition of iMSNs (hM4Di) or excitation of dMSNs (hM3Dq) shifts choices towards
> the immediate small reward.

In [ ]:
sig_counts = {}
for g in ("Inhibitory DREADD", "Excitatory DREADD"):
    _, _, tab_g, _ = report("DREADD", dr[g], label=f"{g}, pre against CNO")
    s = tab_g[(tab_g.delay_s > 0) & (tab_g["diff"] < 0) & (tab_g.p_bonferroni < 0.05)]
    sig_counts[g] = s.delay_s.tolist()
    print(f"  {g}: of 3, 6 and 9 s, lower with adjusted p < 0.05: {sig_counts[g]}")

**Result: partially reproduced.** hM3Dq mice choose the large reward less often on CNO at 3, 6 and 9 s (by 18.0, 17.0 and 12.4 points; adjusted p = 0.035, 0.0024 and 0.016). hM4Di mice shift in the same direction (by 4.2 to 11.6 points; condition F(1,11) = 6.37, p = 0.028) but no single delay reaches significance (adjusted p 0.16 to 0.81), where the paper reports 3, 6 and 9 s.

## Claim fig5jl-dreadd-a-k-auc

> CNO increases K and decreases AUC in hM4Di and hM3Dq mice, reduces A only in hM3Dq mice, and
> changes none of them in mCherry mice.

In [ ]:
expected_pattern = {"mCherry": {"A": 0, "K": 0, "AUC": 0},
                    "Inhibitory DREADD": {"A": 0, "K": 1, "AUC": -1},
                    "Excitatory DREADD": {"A": -1, "K": 1, "AUC": -1}}
matches = 0
for g, pattern in expected_pattern.items():
    fb = icd.fit_table(icd.animal_curves(beh["DREADD"], dr[g], "baseline").dropna())
    fc = icd.fit_table(icd.animal_curves(beh["DREADD"], dr[g], "drug").dropna())
    both = fb.dropna().index.intersection(fc.dropna().index)
    for m, want in pattern.items():
        x, y = fb.loc[both, m], fc.loc[both, m]
        p = stats.wilcoxon(x, y).pvalue
        got = 0 if p >= 0.05 else int(np.sign(np.median(y - x)))
        matches += got == want
        print(f"{g:18s} {m:3s}: n = {len(both)}, median pre {x.median():.3f}, CNO {y.median():.3f}, "
              f"Wilcoxon p = {p:.3g}; found {got:+d}, paper {want:+d}")
print(f"matches: {matches} of 9")

**Result: partially reproduced.** Eight of the nine tests match the paper: mCherry shows no change in A, K or AUC (p 0.36 to 0.90); hM3Dq lowers A (p = 0.012), raises K (p = 0.0015) and lowers the AUC (p = 0.00049); hM4Di lowers the AUC (p = 0.043) and leaves A unchanged (p = 0.20). The exception is K in hM4Di mice (median 0.123 to 0.136, p = 0.20), which the paper reports as higher (p = 0.02).

## Claim fig6bc-ppx-progression

> In parkinsonian mice the effect of PPX on delay discounting grows from the first to the
> fourth session (p < 0.0081), while in control mice it does not (p > 0.99).

In [ ]:
prog = {}
for label, ids in (("Park", park_ids), ("Ctrl", ctrl_ids)):
    a1, a4, tab14, an = report("PPX", ids, "drug", "drug", [1], [4], label=f"{label}, first against fourth PPX session")
    prog[label] = (an["condition"][3], (a4.mean() - a1.mean()).mean())
    print(f"  {label}: session effect p = {an['condition'][3]:.3g}; mean change fourth minus first {prog[label][1]:.2f} points")
park_ok = prog["Park"][0] < 0.05 and prog["Park"][1] < 0
ctrl_ok = prog["Ctrl"][0] >= 0.05
print(f"Park part holds: {park_ok}; Ctrl part holds: {ctrl_ok}")
_ = report("PPX", [a for a in ctrl_ids if a not in P_ANIMALS], "drug", "drug", [1], [4], label="variant, Ctrl without P animals")

**Result: not reproduced.** In 27 parkinsonian mice with a first and a fourth PPX session the choice is lower in the fourth by 5.4 points on average, but the session effect is not significant (F(1,26) = 4.10, p = 0.053; per delay, adjusted p 0.052 to 1). In control mice it does not change (8 of 16 with both sessions, p = 0.79), as the paper reports.

## Claim fig6de-a77636-no-progression

> With A77636 the increase in impulsivity does not change from the first to the fourth session
> (parkinsonian p = 0.59, control p = 0.49).

In [ ]:
for label, ids in (("Park", a_park), ("Ctrl", a_ctrl)):
    a1, a4, _, an = report("A77636", ids, "drug", "drug", [1], [4], label=f"{label}, first against fourth A77636 session")
    print(f"  {label}: session effect p = {an['condition'][3]:.3g}")

**Result: reproduced.** Neither group changes between the first and the fourth A77636 session (parkinsonian, 16 mice, p = 0.71; control, 10 mice, p = 0.42).

## Summary

| Claim | Data | Outcome |
|---|---|---|
| fig4bc-baseline-rates-similar | dandiset | blocked: data missing |
| fig4dg-ppx-bidirectional-identified | dandiset | blocked: data missing |
| fig4hk-modulation-stronger-park | dandiset | blocked: data missing |
| figs5d-msn-response-types | dandiset | blocked: data missing |
| figs5ef-saline-no-change | dandiset | blocked: data missing |
| fig6fg-response-types-shift | dandiset | blocked: data missing |
| fig6hi-increase-type-stronger | dandiset | blocked: data missing |
| mua-ppx-park-vs-ctrl | dandiset | partially reproduced |
| mua-saline-no-change | dandiset | blocked: other (1 of 4 sessions covers the windows) |
| mua-response-shift-first-fourth | dandiset | partially reproduced |
| fig1b-th-depletion | Zenodo | reproduced (one of three background readings) |
| fig1c-rotarod | Zenodo | reproduced |
| fig1fg-baseline-similar | Zenodo | reproduced |
| fig1f-ctrl-ppx-no-change | Zenodo | reproduced (10 of 16 mice) |
| fig1g-park-ppx-impulsive | Zenodo | reproduced |
| fig1ik-a-k-auc | Zenodo | partially reproduced |
| figs2ik-washout | Zenodo | partially reproduced |
| fig2d-ctrl-a77636-no-change | Zenodo | reproduced |
| fig2e-park-a77636-impulsive | Zenodo | reproduced |
| fig2fh-a77636-a-k-auc | Zenodo | reproduced |
| fig2bc-a77636-locomotion | Zenodo | blocked: metadata insufficient |
| fig3a-a-correlates-dms | Zenodo | not reproduced |
| fig3b-k-no-correlation | Zenodo | reproduced |
| fig5g-mcherry-no-change | Zenodo | reproduced |
| fig5hi-dreadd-impulsive | Zenodo | partially reproduced |
| fig5jl-dreadd-a-k-auc | Zenodo | partially reproduced |
| fig5cf-dreadd-locomotion | Zenodo | blocked: metadata insufficient |
| fig6bc-ppx-progression | Zenodo | not reproduced |
| fig6de-a77636-no-progression | Zenodo | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The dandiset has no sorted units with cell types or optogenetic labels, so the paper's unit-level electrophysiology is not tested. The three mua- claims test the same effects on the threshold crossings of each channel, a mixture of cells of all types; their outcomes say what multi-unit activity shows, not what the paper's identified cells show.
- The injection time is inferred from the recording segments and the subject description; the acute drop in firing just after it (median log ratio -0.59 in PPX sessions, against 0.05 at shifted times) supports it in every PPX session with usable threshold crossings except E5 day 4, whose largest drop comes at 80 min; that session is left out of the main analysis.
- Comparisons at each delay use paired t-tests with Bonferroni correction; Prism's post hoc after the two-way ANOVA uses a pooled error term, which can give somewhat different p-values.
- Session files are assigned to conditions by their names, and each animal's sessions in a condition are averaged; the paper's intermediate spreadsheets that list which sessions were used are not deposited.
- The histology background rows are ambiguous (four "blank" and four "background" rows per slice); reading (b) gives the paper's values, the others lower ones.

What was found about the dandiset:

- Sorted units exist in 2 of 56 files and carry no electrode, cell type, MSN classification or optogenetic label; the other 54 files hold unsorted threshold crossings. No file holds laser pulse times, injection times, trials or behaviour, so the paper's Figures 4, 6F-K and S5 cannot be recomputed from the dandiset.
- Two files (C3-2 day 1, E5 day 2) have threshold-crossing timestamps offset by about 30 h from their recordings, and in C9 day 1 they stop at 42 min of a 414-min recording.
- Three sessions have start times in 2025, after the paper was submitted (H0-2 day 3 and day 4, H4-3 day 3), and H0-2 day 4 starts before day 3.
- Electrode locations are "unknown" in every file; genotypes are spelled three ways; subject descriptions contain mis-encoded characters.
- The dandiset's genotype counts do not match Figure 4: 2 control A2a-Cre mice where Figure 4C has 4 control animals with iMSNs.
- Three of the four saline recordings end before the 240-300 min window the paper uses.
- On Zenodo, the six control mice P1-P7 were tested with delays labelled 0, 6, 12 and 18 s (most of their sessions have only three free-choice blocks); the rotarod sheet holds 40 control mice where the paper reports 31; and 61 histology rows have RawIntDen inconsistent with Area x Mean.

In [ ]:
print(f"total run time {(time.time() - T_START) / 60:.1f} min")